# 04 - Machine learning extension (gradient boosting) and what deep learning could add

## Why try ML at all
The HAR model is linear in a handful of features. Attention could matter **non-linearly**: e.g. only *spikes* (a huge jump in views
around earnings) move volatility, while ordinary daily noise in views does not. A linear model averages the two and may miss it. Tree ensembles
can find thresholds and interactions automatically.

## Setup
- **Model:** `HistGradientBoostingRegressor` (depth 3, 150 trees, lr 0.05) - gradient-boosted decision trees.
- **Same discipline:** expanding window, refit every 63 days (cheaper than 21), features lagged.
- **Richer attention features:** besides the two lags, the 5-day mean of views changes and the 60-day z-score of log views (a "spike" indicator).
- **Comparison:** GBM(baseline features) vs GBM(baseline + attention features), plus the OLS HAR as an extra reference.
- **Test:** Diebold-Mariano with HAC only. Clark-West assumes nested *linear* models; with trees its adjustment is not justified, so
  we do **not** use it here (it would give spuriously significant p-values: a GBM that is worse in MSE can still get a small Clark-West p).

In [ ]:
import os, sys, warnings
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from src import model_pipeline as mp
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": .3})
START, END = "2016-01-01", "2026-10-31"

In [ ]:
rows = []
for t, p in mp.PAIRS:
    X, y = mp.build(t, p, START, END, rich=True)
    attn = mp.BASE + mp.ATTN + mp.RICH
    gb_b = mp.walk_forward(X, y, mp.BASE, "gbm", refit=63)
    gb_a = mp.walk_forward(X, y, attn, "gbm", refit=63)
    ols_b = mp.walk_forward(X, y, mp.BASE, "ols", refit=63)
    yt = y.loc[gb_b.index]
    res, _ = mp.compare(y, gb_b, gb_a)
    rows.append(dict(ticker=t, page=p, mse_har_ols=((yt - ols_b)**2).mean(), mse_gbm_base=((yt - gb_b)**2).mean(),
                     mse_gbm_attn=((yt - gb_a)**2).mean(), r2_gain_gbm=res["r2_oos_gain"], dm_stat=res["dm_stat"], p_dm=res["p_dm"]))
ml = pd.DataFrame(rows)
from statsmodels.stats.multitest import multipletests
ml["p_dm_holm"] = multipletests(ml.p_dm, method="holm")[1]
ml.round(4).to_csv("results/ml_results.csv", index=False)
ml.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
w = .27; x = np.arange(len(ml))
ax[0].bar(x - w, ml.mse_har_ols, w, label="HAR (OLS)"); ax[0].bar(x, ml.mse_gbm_base, w, label="GBM, baseline feat.")
ax[0].bar(x + w, ml.mse_gbm_attn, w, label="GBM + attention"); ax[0].set_xticks(x); ax[0].set_xticklabels(ml.ticker)
ax[0].set_title("Out-of-sample MSE (lower is better)"); ax[0].legend(fontsize=8)
ax[0].set_ylim(ml.mse_har_ols.min() * .9, ml[["mse_gbm_base", "mse_gbm_attn"]].values.max() * 1.05)
ax[1].bar(ml.ticker, 100 * ml.r2_gain_gbm, color=np.where(ml.r2_gain_gbm > 0, "tab:green", "tab:red"))
ax[1].axhline(0, color="k", lw=.8); ax[1].set_title("GBM: OOS R2 gain from attention features (%)")
plt.tight_layout(); plt.show()

**Reading (see the table above for the exact numbers).**
1. The **GBM baseline is worse than the simple linear HAR**: with ~2,000 daily observations and strongly persistent,
   nearly linear dynamics, extra flexibility mostly fits noise. This is a known pattern in volatility forecasting.
2. Attention features do not give a significant gain to the GBM either (check `p_dm_holm`).
3. So the null result is **not just an artifact of using a linear model**, for this feature set and these tree hyper-parameters.
   Caveat: no tuning was done; tuning on the same data we test on would reintroduce look-ahead (it needs a nested validation split).

## Where deep learning could go (not run here, with reasons)
| Idea | Why it might help | Why it is not done here |
|---|---|---|
| **LSTM / GRU / temporal CNN** on the raw sequence of returns and views | Learns long-memory and non-linear dynamics without hand-built lags | ~2,000 daily points per ETF is tiny for a neural net; published gains over HAR are small and fragile. Needs a pooled panel of many ETFs/articles. |
| **Pooled panel model** (many ETF-article pairs, one network with ticker embeddings) | Multiplies the sample size, tests the *average* attention effect | Needs a larger, curated pair list; natural next step. |
| **NLP on article *edits* or news text** instead of view counts | Edit content and sentiment are richer than visits | Different data source, much larger engineering effort. |
| **Intraday realized volatility** (5-minute returns) as target | A far less noisy target makes small effects detectable; probably the *single best upgrade* before any deep model | Needs paid/intraday data. |

Rule of thumb applied: **better target and data first, richer model second**. Deep learning on a noisy 5-day volatility target with
2,000 points would mostly add variance.